# imports

In [1]:
from statsmodels.tsa.stattools import adfuller, acf, pacf
from sklearn.ensemble import AdaBoostRegressor, RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import Lasso, Ridge, ElasticNet, LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import xgboost
import lightgbm
import catboost
from sktime.forecasting.arima import AutoARIMA
from sktime.forecasting.fbprophet import Prophet
from sktime.forecasting.croston import Croston
from sktime.forecasting.theta import ThetaForecaster
import optuna
from sklearn.model_selection import TimeSeriesSplit
import numpy as np
import pandas as pd
from pathlib import Path
import pathlib
import math
import holidays
import json
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import os
import random
import torch
torch.set_float32_matmul_precision("high")
import warnings
import xgboost as xgb
import lightgbm as lgb
from neuralforecast import NeuralForecast
from neuralforecast.models import (
    LSTM, GRU, PatchTST, TiDE, TCN, NBEATSx, NHITS, TFT, TSMixerx, KAN
)
from neuralforecast.losses.pytorch import MSE


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

USE_GPU = torch.cuda.is_available()
GPU_DEVICE = "0"
print("GPU available:", USE_GPU)

NF_MODEL_NAMES = {
    "LSTM", "GRU", "PatchTST", "TiDE", "TCN",
    "NBEATSx", "NHITS", "TFT", "TSMixerx", "KAN"
}

# ============================================================
# BASE MODELS
# ============================================================
lasso_model = Lasso(alpha=0.01, max_iter=5000)
xgb_model = xgboost.XGBRegressor(n_jobs=-1,random_state=SEED)
lgb_model = lightgbm.LGBMRegressor(verbose=-1, random_state=SEED, n_jobs=-1)
cat_model = catboost.CatBoostRegressor(verbose=0, thread_count=-1, random_seed=SEED)

rf_model = RandomForestRegressor(n_jobs=-1, random_state=SEED)
gb_model = GradientBoostingRegressor(random_state=SEED)
et_model = ExtraTreesRegressor(n_jobs=-1, random_state=SEED)
ridge_model = Ridge(alpha=1.0)
elasticnet_model = ElasticNet(alpha=0.01, l1_ratio=0.2, max_iter=5000)
linear_model = LinearRegression()
mlp_model = MLPRegressor(
    hidden_layer_sizes=(200, 100, 50),
    activation="relu",
    solver="adam",
    max_iter=1000,
    random_state=SEED
)
svr_model = SVR()
dt_model = DecisionTreeRegressor(random_state=SEED)
auto_arimax_model = AutoARIMA(n_jobs=-1)
prophet_model = Prophet()
croston_model = Croston()
theta_model = ThetaForecaster(deseasonalize=False)

# ============================================================
# HELPERS
# ============================================================
def split_nf_exog_columns(columns):
    """
    Split engineered columns into:
    - historical exogenous: lag features
    - future exogenous: calendar / holiday / cyclical known in advance
    """
    hist_exog = []
    futr_exog = []

    for col in columns:
        if "_lag_" in col:
            hist_exog.append(col)
        else:
            futr_exog.append(col)

    return hist_exog, futr_exog


def make_nf_dataframe(y, X=None, unique_id="series_1"):
    """
    NeuralForecast train dataframe:
    unique_id, ds, y, <exog...>
    """
    df_nf = pd.DataFrame({
        "unique_id": unique_id,
        "ds": pd.to_datetime(y.index),
        "y": y.values
    })

    if X is not None:
        Xc = X.copy()
        Xc.index = pd.to_datetime(Xc.index)
        Xc = Xc.reset_index(drop=False)
        Xc = Xc.rename(columns={Xc.columns[0]: "ds"})
        df_nf = df_nf.merge(Xc, on="ds", how="left")

    return df_nf


def make_nf_future_dataframe(X_future=None, unique_id="series_1", futr_exog_cols=None):
    """
    NeuralForecast future dataframe for predict().
    Only future-known exogenous variables should be included.
    """
    if X_future is None or futr_exog_cols is None or len(futr_exog_cols) == 0:
        return None

    futr_df = pd.DataFrame({
        "unique_id": unique_id,
        "ds": pd.to_datetime(X_future.index)
    })

    for col in futr_exog_cols:
        futr_df[col] = X_future[col].values

    return futr_df


def build_nf_model(name, forecast_horizon, X_train, params, seed=42):
    hist_exog, futr_exog = split_nf_exog_columns(X_train.columns)

    common = {
        "h": forecast_horizon,
        "loss": MSE(),
        "input_size": params.get("input_size", forecast_horizon),
        "max_steps": params.get("max_steps", 500),
        "batch_size": params.get("batch_size", 32),
        "learning_rate": params.get("learning_rate", 1e-3),
        "scaler_type": "minmax",
        "random_seed": seed,
        
    }

    exog_kwargs = {}
    if len(hist_exog) > 0:
        exog_kwargs["hist_exog_list"] = hist_exog
    if len(futr_exog) > 0:
        exog_kwargs["futr_exog_list"] = futr_exog

    if name == "LSTM":
        return LSTM(
            **common,
            **exog_kwargs,
            encoder_n_layers=params.get("encoder_n_layers", 2),
            encoder_hidden_size=params.get("encoder_hidden_size", 128),
            encoder_dropout=params.get("encoder_dropout", 0.0),
            decoder_hidden_size=params.get("decoder_hidden_size", 128),
            decoder_layers=params.get("decoder_layers", 1),
            context_size=params.get("context_size", 10),
        )

    elif name == "GRU":
        return GRU(
            **common,
            **exog_kwargs,
            encoder_n_layers=params.get("encoder_n_layers", 2),
            encoder_hidden_size=params.get("encoder_hidden_size", 128),
            encoder_dropout=params.get("encoder_dropout", 0.0),
            decoder_hidden_size=params.get("decoder_hidden_size", 128),
            decoder_layers=params.get("decoder_layers", 1),
            context_size=params.get("context_size", 10),
        )

    elif name == "PatchTST":
        return PatchTST(
            **common,
            **exog_kwargs,
            hidden_size=params.get("hidden_size", 64),
            n_heads=params.get("n_heads", 4),
            patch_len=params.get("patch_len", 16),
            stride=params.get("stride", 8),
            dropout=params.get("dropout", 0.0),
            revin=params.get("revin", True),
        )

    elif name == "TCN":
        return TCN(
            **common,
            **exog_kwargs,
            kernel_size=params.get("kernel_size", 3),
            dilations=params.get("dilations", [1, 2, 4, 8]),
            encoder_hidden_size=params.get("encoder_hidden_size", 64),
            context_size=params.get("context_size", 10),
        )

    elif name == "TiDE":
        return TiDE(
            **common,
            **exog_kwargs,
            hidden_size=params.get("hidden_size", 256),
            num_encoder_layers=params.get("num_encoder_layers", 2),
            num_decoder_layers=params.get("num_decoder_layers", 2),
            dropout=params.get("dropout", 0.0),
        )

    elif name == "NBEATSx":
        return NBEATSx(
            **common,
            **exog_kwargs,
            stack_types=params.get("stack_types", ["trend", "seasonality", "identity"]),
            n_blocks=params.get("n_blocks", [1, 1, 1]),
            mlp_units=params.get("mlp_units", 3 * [[256, 256]]),
            dropout_prob_theta=params.get("dropout_prob_theta", 0.0),
        )

    elif name == "NHITS":
        return NHITS(
            **common,
            **exog_kwargs,
            n_blocks=params.get("n_blocks", [1, 1, 1]),
            mlp_units=params.get("mlp_units", 3 * [[256, 256]]),
            dropout_prob_theta=params.get("dropout_prob_theta", 0.0),
        )

    elif name == "TFT":
        return TFT(
            **common,
            **exog_kwargs,
            hidden_size=params.get("hidden_size", 64),
            n_head=params.get("n_head", 4),
            dropout=params.get("dropout", 0.1),
        )

    elif name == "TSMixerx":
        return TSMixerx(
            **common,
            **exog_kwargs,
            n_series=1,  # local single-series setup
            n_block=params.get("n_block", 2),
            ff_dim=params.get("ff_dim", 64),
            dropout=params.get("dropout", 0.0),
            revin=params.get("revin", True),
        )

    elif name == "KAN":
        return KAN(
            **common,
            **exog_kwargs,
            hidden_size=params.get("hidden_size", 64),
            n_hidden_layers=params.get("n_hidden_layers", 2),
            grid_size=params.get("grid_size", 5),
            spline_order=params.get("spline_order", 3),
        )

    else:
        raise ValueError(f"Unsupported NeuralForecast model: {name}")


def nf_objective_factory(model_name, X_train, y_train, seed=42):
    def objective(trial):
        tscv = TimeSeriesSplit(n_splits=3)
        mse_scores = []

        for train_idx, val_idx in tscv.split(X_train):
            y_train_fold = y_train.iloc[train_idx].copy()
            X_train_fold = X_train.iloc[train_idx].copy()
            y_val_fold = y_train.iloc[val_idx].copy()
            X_val_fold = X_train.iloc[val_idx].copy()

            h = len(y_val_fold)
            train_len = len(y_train_fold)
            max_input_size = min(10 * h, train_len - 1)

            if max_input_size < h:
                return float("inf")

            params = {
                "input_size": trial.suggest_int("input_size", h, max(h, max_input_size)),
                "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
                "batch_size": trial.suggest_categorical("batch_size", [16, 32, 64]),
                "max_steps": 500,
            }

            if model_name in ["LSTM", "GRU"]:
                params.update({
                    "encoder_n_layers": trial.suggest_int("encoder_n_layers", 1, 4),
                    "encoder_hidden_size": trial.suggest_int("encoder_hidden_size", 32, 256, step=32),
                    "encoder_dropout": trial.suggest_float("encoder_dropout", 0.0, 0.5),
                    "decoder_hidden_size": trial.suggest_int("decoder_hidden_size", 32, 256, step=32),
                    "decoder_layers": trial.suggest_int("decoder_layers", 1, 3),
                    "context_size": trial.suggest_int("context_size", 4, 32),
                })

            elif model_name == "PatchTST":
                params.update({
                    "hidden_size": trial.suggest_int("hidden_size", 32, 128, step=16),
                    "n_heads": trial.suggest_categorical("n_heads", [2, 4, 8]),
                    "patch_len": trial.suggest_categorical("patch_len", [8, 16, 24, 32]),
                    "stride": trial.suggest_categorical("stride", [4, 8, 12, 16]),
                    "dropout": trial.suggest_float("dropout", 0.0, 0.3),
                    "revin": trial.suggest_categorical("revin", [True, False]),
                })

            elif model_name == "TCN":
                params.update({
                    "encoder_hidden_size": trial.suggest_int("encoder_hidden_size", 32, 256, step=32),
                    "decoder_hidden_size": trial.suggest_int("encoder_hidden_size", 32, 256, step=32),
                    "kernel_size": trial.suggest_int("kernel_size", 2, 8),
                    "context_size": trial.suggest_int("context_size", 4, 32),
                })

            elif model_name == "TiDE":
                params.update({
                    "hidden_size": trial.suggest_int("hidden_size", 32, 256, step=32),
                    "dropout": trial.suggest_float("dropout", 0.0, 0.5),
                    "num_encoder_layers": trial.suggest_int("num_encoder_layers", 1, 3),
                    "num_decoder_layers": trial.suggest_int("num_decoder_layers", 1, 3),
                })

            elif model_name in ["NBEATSx", "NHITS"]:
                hidden_1 = trial.suggest_int("hidden_1", 64, 512, step=64)
                hidden_2 = trial.suggest_int("hidden_2", 64, 512, step=64)
                params.update({
                    "mlp_units": 3 * [[hidden_1, hidden_2]],
                    "n_blocks": [1, 1, 1],
                    "dropout_prob_theta": trial.suggest_float("dropout_prob_theta", 0.0, 0.3),
                })

            elif model_name == "TFT":
                params.update({
                    "hidden_size": trial.suggest_int("hidden_size", 16, 128, step=16),
                    "n_head": trial.suggest_categorical("n_head", [2, 4, 8]),
                    "dropout": trial.suggest_float("dropout", 0.0, 0.5),
                })

            elif model_name == "TSMixerx":
                params.update({
                    "n_block": trial.suggest_int("n_block", 1, 4),
                    "ff_dim": trial.suggest_int("ff_dim", 32, 256, step=32),
                    "dropout": trial.suggest_float("dropout", 0.0, 0.3),
                    "revin": trial.suggest_categorical("revin", [True, False]),
                })

            elif model_name == "KAN":
                params.update({
                    "hidden_size": trial.suggest_int("hidden_size", 32, 256, step=32),
                    "n_hidden_layers": trial.suggest_int("n_hidden_layers", 1, 4),
                    "grid_size": trial.suggest_int("grid_size", 3, 8),
                    "spline_order": trial.suggest_int("spline_order", 2, 4),
                })

            try:
                hist_exog_cols, futr_exog_cols = split_nf_exog_columns(X_train_fold.columns)

                train_df_nf = make_nf_dataframe(
                    y_train_fold,
                    X_train_fold,
                    unique_id="series_1"
                )

                futr_df = make_nf_future_dataframe(
                    X_future=X_val_fold,
                    unique_id="series_1",
                    futr_exog_cols=futr_exog_cols
                )

                model = build_nf_model(
                    name=model_name,
                    forecast_horizon=h,
                    X_train=X_train_fold,
                    params=params,
                    seed=seed
                )

                nf = NeuralForecast(models=[model], freq="5min")
                nf.fit(df=train_df_nf)

                if futr_df is not None:
                    preds = nf.predict(futr_df=futr_df)
                else:
                    preds = nf.predict()

                pred_col = preds.columns.difference(["unique_id", "ds"])[0]
                y_pred_fold = preds[pred_col].values

                if len(y_pred_fold) != len(y_val_fold):
                    return float("inf")

                if np.isnan(y_pred_fold).any() or np.isnan(y_val_fold.to_numpy()).any():
                    return float("inf")

                mse_scores.append(mean_squared_error(y_val_fold.to_numpy(), y_pred_fold))

                ######## CLEANUP TO open up RAM and GPU memory
                
                del model
                import gc
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            except Exception:
                return float("inf")

        return float(np.mean(mse_scores))

    return objective


def hpo_models(X_train, y_train, models, opt_trials):
    parameters = {}

    for opt_model in models.keys():
        if opt_model == "Lasso":
            def objective_Lasso(trial):
                sugg_alpha = trial.suggest_float("alpha", 1e-5, 1)
                sugg_fit_intercept = trial.suggest_categorical("fit_intercept", [True, False])
                sugg_selection = trial.suggest_categorical("selection", ["cyclic", "random"])

                X_scaler = MinMaxScaler()
                y_scaler = MinMaxScaler()

                X_train_scaled = X_scaler.fit_transform(X_train)
                y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()

                model = Lasso(
                    alpha=sugg_alpha,
                    fit_intercept=sugg_fit_intercept,
                    selection=sugg_selection,
                    max_iter=3000,
                    random_state=SEED
                )

                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train_scaled):
                    X_train_fold, X_val_fold = X_train_scaled[train_idx], X_train_scaled[val_idx]
                    y_train_fold, y_val_fold = y_train_scaled[train_idx], y_train_scaled[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred_scaled = model.predict(X_val_fold)

                    y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
                    y_val_original = y_scaler.inverse_transform(y_val_fold.reshape(-1, 1)).ravel()

                    mse_scores.append(mean_squared_error(y_val_original, y_pred))



                return np.mean(mse_scores)

            study = optuna.create_study(direction="minimize")
            study.optimize(objective_Lasso, n_trials=opt_trials)
            parameters[opt_model] = study.best_params

        elif opt_model == "XGBoost":
            def objective_XGB(trial):
                params = {
                    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
                    "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
                    "max_depth": trial.suggest_int("max_depth", 3, 10),
                    "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
                    "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                    "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                }

                model = xgboost.XGBRegressor(**params, n_jobs=-1, random_state=SEED)

                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train):
                    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
                    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred = model.predict(X_val_fold)
                    mse_scores.append(mean_squared_error(y_val_fold, y_pred))

                return np.mean(mse_scores)

            study = optuna.create_study(direction="minimize")
            study.optimize(objective_XGB, n_trials=opt_trials)
            parameters[opt_model] = study.best_params

        elif opt_model == "LightGBM":
            def objective_LGBM(trial):
                params = {
                    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
                    "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
                    "max_depth": trial.suggest_int("max_depth", 3, 10),
                    "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
                    "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                    "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                }

                model = lightgbm.LGBMRegressor(**params, verbose=-1, n_jobs=-1, random_state=SEED)

                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train):
                    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
                    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred = model.predict(X_val_fold)
                    mse_scores.append(mean_squared_error(y_val_fold, y_pred))

                return np.mean(mse_scores)

            study = optuna.create_study(direction="minimize")
            study.optimize(objective_LGBM, n_trials=opt_trials)
            parameters[opt_model] = study.best_params

        elif opt_model == "CatBoost":
            def objective_CatBoost(trial):
                params = {
                    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                    "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
                    "depth": trial.suggest_int("depth", 3, 12),
                    "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10, log=True),
                    "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
                }

                model = catboost.CatBoostRegressor(
                    early_stopping_rounds=50,
                    **params,
                    verbose=0,
                    thread_count=-1,
                    random_seed=SEED
                )

                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train):
                    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
                    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred = model.predict(X_val_fold)
                    mse_scores.append(mean_squared_error(y_val_fold, y_pred))

                return np.mean(mse_scores)

            study = optuna.create_study(direction="minimize")
            study.optimize(objective_CatBoost, n_trials=opt_trials)
            parameters[opt_model] = study.best_params

        elif opt_model == "RandomForest":
            def objective_RF(trial):
                params = {
                    "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
                    "max_depth": trial.suggest_int("max_depth", 3, 30),
                    "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50),
                    "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
                    "max_features": trial.suggest_float("max_features", 0.3, 1.0),
                    "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
                }

                model = RandomForestRegressor(**params, n_jobs=-1, random_state=SEED)

                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train):
                    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
                    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred = model.predict(X_val_fold)
                    mse_scores.append(mean_squared_error(y_val_fold, y_pred))

                return np.mean(mse_scores)

            study = optuna.create_study(direction="minimize")
            study.optimize(objective_RF, n_trials=opt_trials)
            parameters[opt_model] = study.best_params

        elif opt_model == "GradientBoosting":
            def objective_GBR(trial):
                params = {
                    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
                    "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
                    "max_depth": trial.suggest_int("max_depth", 3, 10),
                    "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
                    "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50),
                    "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                    "max_features": trial.suggest_float("max_features", 0.3, 1.0),
                }

                model = GradientBoostingRegressor(**params,n_jobs=-1, random_state=SEED)

                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train):
                    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
                    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred = model.predict(X_val_fold)
                    mse_scores.append(mean_squared_error(y_val_fold, y_pred))

                return np.mean(mse_scores)

            study = optuna.create_study(direction="minimize")
            study.optimize(objective_GBR, n_trials=opt_trials)
            parameters[opt_model] = study.best_params

        elif opt_model == "ExtraTrees":
            def objective_ETR(trial):
                params = {
                    "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
                    "max_depth": trial.suggest_int("max_depth", 3, 30),
                    "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
                    "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50),
                    "max_features": trial.suggest_float("max_features", 0.3, 1.0),
                    "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
                }

                model = ExtraTreesRegressor(**params, n_jobs=-1,random_state=SEED)

                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train):
                    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
                    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred = model.predict(X_val_fold)
                    mse_scores.append(mean_squared_error(y_val_fold, y_pred))

                return np.mean(mse_scores)

            study = optuna.create_study(direction="minimize")
            study.optimize(objective_ETR, n_trials=opt_trials)
            parameters[opt_model] = study.best_params

        elif opt_model == "Ridge":
            def objective_Ridge(trial):
                params = {
                    "alpha": trial.suggest_float("alpha", 1e-5, 10, log=True),
                    "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False])
                }

                X_scaler = MinMaxScaler()
                y_scaler = MinMaxScaler()

                X_train_scaled = X_scaler.fit_transform(X_train)
                y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()

                model = Ridge(**params)

                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train_scaled):
                    X_train_fold, X_val_fold = X_train_scaled[train_idx], X_train_scaled[val_idx]
                    y_train_fold, y_val_fold = y_train_scaled[train_idx], y_train_scaled[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred_scaled = model.predict(X_val_fold)

                    y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
                    y_val_original = y_scaler.inverse_transform(y_val_fold.reshape(-1, 1)).ravel()

                    mse_scores.append(mean_squared_error(y_val_original, y_pred))

                return np.mean(mse_scores)

            study = optuna.create_study(direction="minimize")
            study.optimize(objective_Ridge, n_trials=opt_trials)
            parameters[opt_model] = study.best_params

        elif opt_model == "ElasticNet":
            def objective_ElasticNet(trial):
                params = {
                    "alpha": trial.suggest_float("alpha", 1e-5, 10.0, log=True),
                    "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),
                    "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False]),
                }

                X_scaler = MinMaxScaler()
                y_scaler = MinMaxScaler()

                X_train_scaled = X_scaler.fit_transform(X_train)
                y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()

                model = ElasticNet(**params)

                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train_scaled):
                    X_train_fold, X_val_fold = X_train_scaled[train_idx], X_train_scaled[val_idx]
                    y_train_fold, y_val_fold = y_train_scaled[train_idx], y_train_scaled[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred_scaled = model.predict(X_val_fold)

                    y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
                    y_val_original = y_scaler.inverse_transform(y_val_fold.reshape(-1, 1)).ravel()

                    mse_scores.append(mean_squared_error(y_val_original, y_pred))

                return np.mean(mse_scores)

            study = optuna.create_study(direction="minimize")
            study.optimize(objective_ElasticNet, n_trials=opt_trials)
            parameters[opt_model] = study.best_params

        elif opt_model == "SVR":
            def objective_SVR(trial):
                params = {
                    "C": trial.suggest_float("C", 1e-3, 100.0, log=True),
                    "epsilon": trial.suggest_float("epsilon", 1e-4, 1.0, log=True),
                    "kernel": trial.suggest_categorical("kernel", ["linear", "poly", "rbf", "sigmoid"]),
                    "gamma": trial.suggest_categorical("gamma", ["scale", "auto"]),
                    "shrinking": trial.suggest_categorical("shrinking", [True, False]),
                }

                if params["kernel"] == "poly":
                    params["degree"] = trial.suggest_int("degree", 2, 5)

                X_scaler = MinMaxScaler()
                y_scaler = MinMaxScaler()

                X_train_scaled = X_scaler.fit_transform(X_train)
                y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()

                model = SVR(**params)

                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train_scaled):
                    X_train_fold, X_val_fold = X_train_scaled[train_idx], X_train_scaled[val_idx]
                    y_train_fold, y_val_fold = y_train_scaled[train_idx], y_train_scaled[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred_scaled = model.predict(X_val_fold)

                    y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
                    y_val_original = y_scaler.inverse_transform(y_val_fold.reshape(-1, 1)).ravel()

                    mse_scores.append(mean_squared_error(y_val_original, y_pred))

                return np.mean(mse_scores)

            study = optuna.create_study(direction="minimize")
            study.optimize(objective_SVR, n_trials=opt_trials)
            parameters[opt_model] = study.best_params

        elif opt_model == "DecisionTree":
            def objective_DTR(trial):
                params = {
                    "max_depth": trial.suggest_int("max_depth", 3, 30),
                    "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
                    "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50),
                    "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
                    "splitter": trial.suggest_categorical("splitter", ["best", "random"]),
                }

                model = DecisionTreeRegressor(**params, random_state=SEED)

                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train):
                    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
                    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred = model.predict(X_val_fold)
                    mse_scores.append(mean_squared_error(y_val_fold, y_pred))

                return np.mean(mse_scores)

            study = optuna.create_study(direction="minimize")
            study.optimize(objective_DTR, n_trials=opt_trials)
            parameters[opt_model] = study.best_params

        elif opt_model == "MLP":
            def objective_MLP(trial):
                num_layers = trial.suggest_int("num_layers", 2, 4)
                hidden_layer_sizes = tuple(
                    trial.suggest_int(f"layer_{i+1}", 50, 500, step=50) for i in range(num_layers)
                )

                params = {
                    "hidden_layer_sizes": hidden_layer_sizes,
                    "activation": trial.suggest_categorical("activation", ["identity", "logistic", "tanh", "relu"]),
                    "alpha": trial.suggest_float("alpha", 1e-5, 1e-1, log=True),
                    "learning_rate": trial.suggest_categorical("learning_rate", ["constant", "invscaling", "adaptive"]),
                    "learning_rate_init": trial.suggest_float("learning_rate_init", 1e-4, 1e-1, log=True),
                    "batch_size": trial.suggest_categorical("batch_size", ["auto", 32, 64, 128]),
                }

                X_scaler = MinMaxScaler()
                y_scaler = MinMaxScaler()

                X_train_scaled = X_scaler.fit_transform(X_train)
                y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()

                model = MLPRegressor(**params, max_iter=300, random_state=SEED)

                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train_scaled):
                    X_train_fold, X_val_fold = X_train_scaled[train_idx], X_train_scaled[val_idx]
                    y_train_fold, y_val_fold = y_train_scaled[train_idx], y_train_scaled[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred_scaled = model.predict(X_val_fold)

                    y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
                    y_val_original = y_scaler.inverse_transform(y_val_fold.reshape(-1, 1)).ravel()

                    mse_scores.append(mean_squared_error(y_val_original, y_pred))

                return np.mean(mse_scores)

            study = optuna.create_study(direction="minimize")
            study.optimize(objective_MLP, n_trials=opt_trials)

            best_params = study.best_params
            num_layers = best_params.pop("num_layers")
            hidden_layer_sizes = tuple(best_params.pop(f"layer_{i+1}") for i in range(num_layers))
            best_params["hidden_layer_sizes"] = hidden_layer_sizes
            parameters[opt_model] = best_params

        elif opt_model == "Prophet":
            def objective_prophet_ext(trial):
                params = {
                    "seasonality_mode": trial.suggest_categorical("seasonality_mode", ["additive", "multiplicative"]),
                    "seasonality_prior_scale": trial.suggest_float("seasonality_prior_scale", 0.01, 10.0),
                    "changepoint_prior_scale": trial.suggest_float("changepoint_prior_scale", 0.001, 0.5),
                    "holidays_prior_scale": trial.suggest_float("holidays_prior_scale", 0.01, 10.0),
                }

                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train):
                    y_train_fold = y_train.iloc[train_idx]
                    X_train_fold = X_train.iloc[train_idx]
                    y_val_fold = y_train.iloc[val_idx]
                    X_val_fold = X_train.iloc[val_idx]

                    model = Prophet(**params)
                    model.fit(y_train_fold, X=X_train_fold)

                    fh_val = np.arange(1, len(y_val_fold) + 1)
                    y_pred_fold = model.predict(fh=fh_val, X=X_val_fold)

                    mse_scores.append(mean_squared_error(y_val_fold, y_pred_fold))

                return np.mean(mse_scores)

            study = optuna.create_study(direction="minimize")
            study.optimize(objective_prophet_ext, n_trials=opt_trials)
            parameters[opt_model] = study.best_params

        elif opt_model == "Croston":
            def objective_croston(trial):
                params = {
                    "smoothing": trial.suggest_float("smoothing", 0.01, 1.0)
                }

                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(y_train):
                    y_train_fold = y_train.iloc[train_idx]
                    y_val_fold = y_train.iloc[val_idx]

                    model = Croston(**params)
                    model.fit(y_train_fold)

                    fh_val = np.arange(1, len(y_val_fold) + 1)
                    y_pred_fold = model.predict(fh=fh_val)

                    mse_scores.append(mean_squared_error(y_val_fold, y_pred_fold))

                return np.mean(mse_scores)

            study = optuna.create_study(direction="minimize")
            study.optimize(objective_croston, n_trials=opt_trials)
            parameters[opt_model] = study.best_params

        elif opt_model == "Theta":
            def objective_theta(trial):
                if (y_train < 0).any() or (y_train == 0).any():
                    deseasonalize_option = False
                else:
                    deseasonalize_option = trial.suggest_categorical("deseasonalize", [True, False])

                params = {
                    "initial_level": trial.suggest_float("initial_level", 0.01, 1.0),
                    "deseasonalize": deseasonalize_option,
                    "sp": trial.suggest_int("sp", 1, min(36, len(y_train) // 2))
                }

                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(y_train):
                    y_train_fold = y_train.iloc[train_idx]
                    y_val_fold = y_train.iloc[val_idx]

                    if y_train_fold.index.freq is None:
                        y_train_fold.index.freq = pd.infer_freq(y_train_fold.index)

                    model = ThetaForecaster(**params)
                    model.fit(y_train_fold)

                    fh_val = np.arange(1, len(y_val_fold) + 1)
                    y_pred_fold = model.predict(fh=fh_val)

                    mse_scores.append(mean_squared_error(y_val_fold, y_pred_fold))

                return np.mean(mse_scores)

            study = optuna.create_study(direction="minimize")
            study.optimize(objective_theta, n_trials=opt_trials)
            parameters[opt_model] = study.best_params

        elif opt_model in NF_MODEL_NAMES:
            study = optuna.create_study(direction="minimize")
            study.optimize(
                nf_objective_factory(opt_model, X_train, y_train, seed=SEED),
                n_trials=opt_trials
            )
            parameters[opt_model] = study.best_params

    return parameters


def find_best_arima_params(y, max_lag=20):
    def adf_test(series):
        result = adfuller(series, autolag="AIC")
        return 0 if result[1] < 0.05 else 1

    d = adf_test(y)
    y_diff = y.diff(d).dropna() if d > 0 else y
    adjusted_max_lag = min(max_lag, len(y_diff) // 2)

    pacf_values = pacf(y_diff, nlags=adjusted_max_lag)
    p = np.argmax(np.abs(pacf_values) < 1.96 / np.sqrt(len(y_diff)))

    acf_values = acf(y_diff, nlags=adjusted_max_lag)
    q = np.argmax(np.abs(acf_values) < 1.96 / np.sqrt(len(y_diff)))

    return p, d, q


def forecasting_households(day,
                            project_path,
                           df_original,
                           df_weather,
                           date,
                           forecast_end_date,
                           forecast_horizon,
                           training_size,
                           feature_selection,
                           plot_forecast,
                           hyperparameter_opt,
                           models,
                           number_of_houses,
                           opt_trials=30):

    path = pathlib.Path(project_path) / "data"

    df_weather = df_weather.copy()
    df_original = df_original.copy()

    df_weather.index = pd.to_datetime(df_weather.index, utc=True).tz_localize(None)
    df_original.index = pd.to_datetime(df_original.index, format="%d.%m.%Y %H:%M")
    df_original = df_original.sort_index()
    df_original.rename(columns={"0": "avg_power"}, inplace=True)

    target = "avg_power"
    df_original = df_original[[target]]

    initial_date = pd.Timestamp(date)
    end_date = pd.Timestamp(forecast_end_date)
    time_step = (df_original.index[1] - df_original.index[0]).seconds // 60
    step_size = pd.Timedelta(minutes=forecast_horizon * time_step)
    n_iterations = math.ceil((end_date - initial_date) / step_size)

    weather_feature_list = df_weather.columns.to_list()
    all_predictions = []

    for i in range(n_iterations):
        print(f"\nIteration {i+1}/{n_iterations}")

        current_date = initial_date + i * step_size
        df_trafo = df_original[df_original.index < current_date].copy()
        df_trafo = df_trafo.tail(training_size)

        df_trafo.index = pd.to_datetime(df_trafo.index)
        df_weather.index = pd.to_datetime(df_weather.index)

        start_timestamp = df_trafo.index.min()
        end_timestamp = df_weather.index.max()

        df = pd.merge(df_trafo, df_weather, left_index=True, right_index=True, how="outer")
        df = df.loc[start_timestamp:end_timestamp]

        first_nan_index = df[df[target].isna()].index.min()

        if pd.isna(first_nan_index):
            filtered_df = pd.DataFrame()
        else:
            start_position = df.index.get_loc(first_nan_index)
            end_position = min(start_position + forecast_horizon, len(df))
            filtered_df = df.iloc[:end_position].copy()

        # ============================================================
        # FEATURE ENGINEERING
        # ============================================================
        for lag in range(forecast_horizon, forecast_horizon * 2 + 1):
            filtered_df[f"{target}_lag_{lag}"] = df[target].shift(lag)

        for lag in range(1, forecast_horizon + 1):
            for g in weather_feature_list:
                filtered_df[f"{g}_lag_{lag}"] = df[g].shift(lag)

        filtered_df.index = pd.to_datetime(filtered_df.index)

        filtered_df["minute"] = filtered_df.index.minute
        filtered_df["hour"] = filtered_df.index.hour
        filtered_df["day_of_week"] = filtered_df.index.dayofweek
        filtered_df["day_of_year"] = filtered_df.index.dayofyear
        filtered_df["week"] = filtered_df.index.isocalendar().week.astype(int)
        filtered_df["month"] = filtered_df.index.month
        filtered_df["year"] = filtered_df.index.year
        filtered_df["is_weekend"] = (filtered_df["day_of_week"] >= 5).astype(int)

        german_holidays = holidays.Germany()
        filtered_df["holiday"] = filtered_df.index.to_series().apply(
            lambda x: 1 if x in german_holidays else 0
        )

        minute_period = 60
        hour_period = 24
        week_period = 7
        month_period = 12
        year_period = 365.25

        filtered_df["minute_sin"] = np.sin(2 * np.pi * filtered_df["minute"] / minute_period)
        filtered_df["minute_cos"] = np.cos(2 * np.pi * filtered_df["minute"] / minute_period)
        filtered_df["hour_sin"] = np.sin(2 * np.pi * filtered_df["hour"] / hour_period)
        filtered_df["hour_cos"] = np.cos(2 * np.pi * filtered_df["hour"] / hour_period)
        filtered_df["dayofweek_sin"] = np.sin(2 * np.pi * filtered_df["day_of_week"] / week_period)
        filtered_df["dayofweek_cos"] = np.cos(2 * np.pi * filtered_df["day_of_week"] / week_period)
        filtered_df["dayofyear_sin"] = np.sin(2 * np.pi * filtered_df["day_of_year"] / year_period)
        filtered_df["dayofyear_cos"] = np.cos(2 * np.pi * filtered_df["day_of_year"] / year_period)
        filtered_df["week_sin"] = np.sin(2 * np.pi * filtered_df["week"] / week_period)
        filtered_df["week_cos"] = np.cos(2 * np.pi * filtered_df["week"] / week_period)
        filtered_df["month_sin"] = np.sin(2 * np.pi * filtered_df["month"] / month_period)
        filtered_df["month_cos"] = np.cos(2 * np.pi * filtered_df["month"] / month_period)

        filtered_df = filtered_df.dropna(subset=[col for col in filtered_df.columns if col != target])

        split_point = len(filtered_df) - forecast_horizon
        if split_point <= 0:
            print("current_date:", current_date)
            print("len(df_trafo):", len(df_trafo))
            print("len(df):", len(df))
            print("first_nan_index:", first_nan_index)
            print("len(filtered_df) before dropna:", len(filtered_df))

            temp_filtered = filtered_df.dropna(subset=[col for col in filtered_df.columns if col != target])
            print("len(filtered_df) after dropna:", len(temp_filtered))
            print("forecast_horizon:", forecast_horizon)
            raise ValueError("Not enough data to create a training and prediction split!")

        y_train = filtered_df[target].iloc[:split_point]
        y_test = filtered_df[target].iloc[split_point:]

        X_train = filtered_df.drop(columns=[target]).iloc[:split_point]
        X_test = filtered_df.drop(columns=[target]).iloc[split_point:]

        # ============================================================
        # FEATURE SELECTION
        # ============================================================
        if feature_selection:
            correlations = X_train.corrwith(y_train)
            correlations = correlations.replace([np.inf, -np.inf], np.nan).dropna()

            selected_features = correlations[abs(correlations) >= 0.1].index.tolist()

            if len(selected_features) == 0:
                top_k = min(10, len(correlations))
                selected_features = correlations.abs().sort_values(ascending=False).head(top_k).index.tolist()
                print(f"No features passed threshold. Using top {top_k} correlated features.")

            X_train = X_train[selected_features]
            X_test = X_test[selected_features]

        # ============================================================
        # HPO
        # ============================================================
        if hyperparameter_opt:
            opt_parameters = hpo_models(X_train, y_train, models, opt_trials)
        else:
            opt_parameters = {
                "Lasso": {"alpha": 0.01, "fit_intercept": True, "selection": "cyclic"},
                "XGBoost": {
                    "learning_rate": 0.05, "n_estimators": 200, "max_depth": 6,
                    "min_child_weight": 1, "subsample": 0.8, "colsample_bytree": 0.8
                },
                "LightGBM": {
                    "learning_rate": 0.05, "n_estimators": 200, "max_depth": 6,
                    "min_child_weight": 1, "subsample": 0.8, "colsample_bytree": 0.8
                },
                "CatBoost": {
                    "learning_rate": 0.05, "n_estimators": 200, "depth": 6,
                    "l2_leaf_reg": 3.0, "bagging_temperature": 0.0
                },
                "RandomForest": {
                    "n_estimators": 200, "max_depth": 15, "min_samples_split": 2,
                    "min_samples_leaf": 1, "max_features": 0.7, "bootstrap": True
                },
                "GradientBoosting": {
                    "learning_rate": 0.05, "n_estimators": 200, "max_depth": 3,
                    "min_samples_split": 2, "min_samples_leaf": 1,
                    "subsample": 1.0, "max_features": 0.7
                },
                "ExtraTrees": {
                    "n_estimators": 200, "max_depth": 15, "min_samples_split": 2,
                    "min_samples_leaf": 1, "max_features": 0.7, "bootstrap": False
                },
                "Ridge": {"alpha": 1.0, "fit_intercept": True},
                "ElasticNet": {"alpha": 0.01, "l1_ratio": 0.5, "fit_intercept": True},
                "LinearRegression": {},
                "MLP": {
                    "hidden_layer_sizes": (200, 100),
                    "activation": "relu",
                    "alpha": 1e-4,
                    "learning_rate": "adaptive",
                    "learning_rate_init": 1e-3,
                    "batch_size": 64
                },
                "SVR": {"C": 1.0, "epsilon": 0.1, "kernel": "rbf", "gamma": "scale"},
                "DecisionTree": {
                    "max_depth": 10, "min_samples_split": 2,
                    "min_samples_leaf": 1, "max_features": None, "splitter": "best"
                },
                "Prophet": {
                    "seasonality_mode": "additive",
                    "seasonality_prior_scale": 10.0,
                    "changepoint_prior_scale": 0.05,
                    "holidays_prior_scale": 10.0
                },
                "Croston": {"smoothing": 0.1},
                "Theta": {
                    "initial_level": 0.2,
                    "deseasonalize": False,
                    "sp": min(96, max(2, len(y_train) // 10))
                },
                "LSTM": {
                    "input_size": forecast_horizon,
                    "encoder_n_layers": 2,
                    "encoder_hidden_size": 128,
                    "encoder_dropout": 0.0,
                    "context_size": 10,
                    "decoder_hidden_size": 128,
                    "decoder_layers": 1,
                    "learning_rate": 1e-3,
                    "batch_size": 16
                },
                "GRU": {
                    "input_size": forecast_horizon,
                    "encoder_n_layers": 2,
                    "encoder_hidden_size": 128,
                    "encoder_dropout": 0.0,
                    "context_size": 10,
                    "decoder_hidden_size": 128,
                    "decoder_layers": 1,
                    "learning_rate": 1e-3,
                    "batch_size": 16
                },
                "PatchTST": {
                    "input_size": forecast_horizon,
                    "learning_rate": 1e-3,
                    "batch_size": 32,
                    "hidden_size": 64,
                    "n_heads": 4,
                    "patch_len": 16,
                    "stride": 8,
                    "dropout": 0.0,
                    "revin": True
                },
                "TiDE": {
                    "input_size": forecast_horizon,
                    "learning_rate": 1e-3,
                    "batch_size": 32,
                    "hidden_size": 128,
                    "dropout": 0.0,
                    "num_encoder_layers": 2,
                    "num_decoder_layers": 2
                },
                "TCN": {
                    "input_size": forecast_horizon,
                    "learning_rate": 1e-3,
                    "batch_size": 16,
                    "encoder_hidden_size": 128,
                    "decoder_hidden_size": 128,
                    "kernel_size": 2,
                    "dilations": [1, 2, 4, 8, 16],
                    "context_size": 10
                },
                "NBEATSx": {
                    "input_size": forecast_horizon,
                    "learning_rate": 1e-3,
                    "batch_size": 32,
                    "mlp_units": 3 * [[256, 256]],
                    "n_blocks": [1, 1, 1],
                    "dropout_prob_theta": 0.0
                },
                "NHITS": {
                    "input_size": forecast_horizon,
                    "learning_rate": 1e-3,
                    "batch_size": 16,
                    "mlp_units": 3 * [[128, 128]],
                    "n_blocks": [1, 1, 1],
                    "dropout_prob_theta": 0.0
                },
                "TFT": {
                    "input_size": forecast_horizon,
                    "learning_rate": 1e-3,
                    "batch_size": 32,
                    "hidden_size": 64,
                    "n_head": 4,
                    "dropout": 0.1
                },
                "TSMixerx": {
                    "input_size": forecast_horizon,
                    "learning_rate": 1e-3,
                    "batch_size": 32,
                    "n_block": 2,
                    "ff_dim": 64,
                    "dropout": 0.0,
                    "revin": True
                },
                "KAN": {
                    "input_size": forecast_horizon,
                    "learning_rate": 1e-3,
                    "batch_size": 16,
                    "hidden_size": 32,
                    "n_hidden_layers": 2,
                    "grid_size": 3,
                    "spline_order": 3
                },
            }

        # ============================================================
        # REINITIALIZE MODELS
        # ============================================================
        reinitialized_models = {}

        for name, params in opt_parameters.items():
            if name not in models:
                continue

            if name == "Lasso":
                reinitialized_models[name] = Lasso(**params, max_iter=5000)

            elif name == "XGBoost":
                reinitialized_models[name] = xgboost.XGBRegressor(
                    **params, device="cuda", tree_method="hist", random_state=SEED
                )

            elif name == "LightGBM":
                reinitialized_models[name] = lightgbm.LGBMRegressor(
                    **params, device_type="gpu", verbose=-1, random_state=SEED, n_jobs=-1
                )

            elif name == "CatBoost":
                reinitialized_models[name] = catboost.CatBoostRegressor(
                    **params, task_type="GPU", devices=GPU_DEVICE, verbose=0, random_seed=SEED
                )

            elif name == "RandomForest":
                reinitialized_models[name] = RandomForestRegressor(**params, n_jobs=-1, random_state=SEED)

            elif name == "GradientBoosting":
                reinitialized_models[name] = GradientBoostingRegressor(**params, random_state=SEED)

            elif name == "ExtraTrees":
                reinitialized_models[name] = ExtraTreesRegressor(**params, n_jobs=-1, random_state=SEED)

            elif name == "Ridge":
                reinitialized_models[name] = Ridge(**params)

            elif name == "ElasticNet":
                reinitialized_models[name] = ElasticNet(**params, max_iter=5000)

            elif name == "LinearRegression":
                reinitialized_models[name] = LinearRegression()

            elif name == "DecisionTree":
                reinitialized_models[name] = DecisionTreeRegressor(**params, random_state=SEED)

            elif name == "MLP":
                reinitialized_models[name] = MLPRegressor(**params, max_iter=1000, random_state=SEED)

            elif name == "SVR":
                reinitialized_models[name] = SVR(**params)

            elif name == "Prophet":
                reinitialized_models[name] = Prophet(**params)

            elif name == "Croston":
                reinitialized_models[name] = Croston(**params)

            elif name == "Theta":
                reinitialized_models[name] = ThetaForecaster(**params)

            elif name in NF_MODEL_NAMES:
                reinitialized_models[name] = build_nf_model(
                    name=name,
                    forecast_horizon=forecast_horizon,
                    X_train=X_train,
                    params=params,
                    seed=SEED
                )

        models_current = reinitialized_models

        # ============================================================
        # SCALING
        # ============================================================
        scaler = MinMaxScaler()
        scaled_target = MinMaxScaler()

        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        y_train_scaled = scaled_target.fit_transform(y_train.values.reshape(-1, 1)).flatten()

        y_preds = {}

        # ============================================================
        # TRAIN
        # ============================================================
        for name, model in models_current.items():
            print(f"Training {name}...")

            if name in ["Lasso", "Ridge", "ElasticNet", "LinearRegression", "MLP", "SVR"]:
                model.fit(X_train_scaled, y_train_scaled)

            elif name in ["AutoARIMAX", "Prophet"]:
                model.fit(y_train, X=X_train)

            elif name in ["Croston", "Theta"]:
                y_train_local = y_train.copy()
                if y_train_local.index.freq is None:
                    inferred_freq = pd.infer_freq(y_train_local.index)
                    if inferred_freq is None:
                        raise ValueError("Cannot infer a valid frequency for y_train.")
                    y_train_local = y_train_local.asfreq(inferred_freq)

                model.fit(y_train_local)

            elif name in NF_MODEL_NAMES:
                hist_exog_cols, futr_exog_cols = split_nf_exog_columns(X_train.columns)

                train_df_nf = make_nf_dataframe(y_train, X_train, unique_id="series_1")

                nf = NeuralForecast(models=[model], freq="5min")
                nf.fit(df=train_df_nf)

                models_current[name] = {
                    "nf": nf,
                    "futr_exog_cols": futr_exog_cols
                }

            else:
                model.fit(X_train, y_train)

            print(f"{name} training complete.")

        # ============================================================
        # PREDICT
        # ============================================================
        for name, model in models_current.items():
            if name in ["Lasso", "Ridge", "ElasticNet", "LinearRegression", "MLP", "SVR"]:
                y_preds[name] = scaled_target.inverse_transform(
                    model.predict(X_test_scaled).reshape(-1, 1)
                ).flatten()

            elif name in ["AutoARIMAX", "Prophet"]:
                fh = np.arange(1, len(y_test) + 1)
                y_pred_auto = model.predict(fh=fh, X=X_test)
                y_pred_auto.index = y_test.index
                y_preds[name] = y_pred_auto.values

            elif name in ["Croston", "Theta"]:
                fh = np.arange(1, len(y_test) + 1)
                y_pred_univariate = model.predict(fh=fh)
                y_pred_univariate.index = y_test.index
                y_preds[name] = y_pred_univariate.values

            elif name in NF_MODEL_NAMES:
                nf_obj = model["nf"]
                futr_exog_cols = model["futr_exog_cols"]

                futr_df = make_nf_future_dataframe(
                    X_future=X_test,
                    unique_id="series_1",
                    futr_exog_cols=futr_exog_cols
                )

                if futr_df is not None:
                    preds_nf = nf_obj.predict(futr_df=futr_df)
                else:
                    preds_nf = nf_obj.predict()

                pred_col = preds_nf.columns.difference(["unique_id", "ds"])[0]
                preds_nf = preds_nf.copy()
                preds_nf["ds"] = pd.to_datetime(preds_nf["ds"])
                preds_nf = preds_nf.set_index("ds")

                y_pred_nf = preds_nf.loc[y_test.index, pred_col]
                y_preds[name] = y_pred_nf.values


                # FREE MEMORY
                del preds_nf
                del y_pred_nf
                del futr_df
                del nf_obj
                import gc
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            else:
                y_preds[name] = model.predict(X_test)

        y_pred_df = pd.DataFrame(y_preds, index=y_test.index)

        y_actual = df_original.loc[y_test.index, target]

        inferred_freq = pd.infer_freq(df_original.index)
        if inferred_freq:
            freq_timedelta = pd.to_timedelta(inferred_freq)
            freq_minutes = freq_timedelta.total_seconds() / 60
        else:
            raise ValueError("Could not infer frequency from the dataset index.")

        time_offset = pd.Timedelta(minutes=freq_minutes * forecast_horizon)
        y_pred_naive = df_original.loc[y_test.index - time_offset, target]

        mse_scores = {name: mean_squared_error(y_actual, y_preds[name]) for name in y_preds.keys()}
        mse_scores["Naive Forecast"] = mean_squared_error(y_actual, y_pred_naive)

        for name, mse in mse_scores.items():
            print(f"MSE - {name}: {mse:.5f}")

        y_pred_df["Naive Forecast"] = y_pred_naive.values
        y_pred_df["Actual"] = y_actual.values
        all_predictions.append(y_pred_df)

    # ============================================================
    # SAVE PREDICTIONS
    # ============================================================
    final_predictions = pd.concat(all_predictions)

    path = pathlib.Path(project_path) / "data" / "outputs"
    path.mkdir(parents=True, exist_ok=True)

    filename = f"final_predictions_{day}_{number_of_houses}_houses_and_{forecast_horizon}_steps.csv"
    filepath = path / filename
    final_predictions.to_csv(filepath, index=True)

    print(f"CSV saved at: {filepath}")

    filename = f"opt_parameters_{number_of_houses}_houses_{forecast_horizon}_steps.json"
    filepath = path / filename
    with open(filepath, "w") as f:
        json.dump(opt_parameters, f, indent=4)

    print(f"JSON saved at: {filepath}")

    if plot_forecast:
        path = pathlib.Path(project_path) / "data" / "plots"
        path.mkdir(parents=True, exist_ok=True)

        filename = f"actual_vs_predictions_{number_of_houses}_houses_{forecast_horizon}_steps.png"
        filepath = path / filename

        plt.figure(figsize=(12, 6))
        colors = list(mcolors.TABLEAU_COLORS.values())

        first_timestamp = final_predictions.index.min()
        previous_day_start = first_timestamp - pd.Timedelta(days=1)

        previous_day_actual = df_original.loc[previous_day_start:first_timestamp, target]
        current_actual = final_predictions["Actual"]
        actual_combined = pd.concat([previous_day_actual, current_actual])

        plt.plot(actual_combined.index, actual_combined, label="Actual", color="black", linewidth=2)

        for i, column in enumerate(final_predictions.columns):
            if column != "Actual":
                plt.plot(
                    final_predictions.index,
                    final_predictions[column],
                    label=column,
                    color=colors[i % len(colors)]
                )

        plt.legend(loc="upper left", frameon=True, facecolor="white", edgecolor="black")
        plt.xlabel("Time")
        plt.ylabel("Value")
        plt.title("Forecast vs Actual")
        plt.savefig(filepath, dpi=300, bbox_inches="tight")
        plt.close()

        print(f"Plot saved at: {filepath}")

GPU available: True


# start

In [2]:
models = {
#    "Lasso": lasso_model,
    "XGBoost": xgb_model,
    "LightGBM": lgb_model,
    "CatBoost": cat_model,
    "RandomForest": rf_model,
#    "GradientBoosting": gb_model,
#    "ExtraTrees": et_model,
#    "Ridge": ridge_model,
    "ElasticNet": elasticnet_model,
    "MLP": mlp_model,
    "SVR": svr_model,
#    "DecisionTree": dt_model,
#    "LSTM": None,
#    "GRU": None,
#    "PatchTST": None,
#    "TiDE": None,
#    "TCN": None,
#    "NBEATSx": None,
#    "NHITS": None,
#    "TFT": None,
#    "TSMixerx": None,
#    "KAN": None,

}


project_path = r"C:\Users\CR58XM\Desktop\Slovak_case_study"
path = pathlib.Path(project_path) / "outputs"

first_set_of_house = 10
max_number_of_houses = 10

df_weather = pd.read_csv(path / "weather.csv", index_col=0, parse_dates=[0])

days=["day1","day2","day3","day4","day5","day6"]

for day in days:
    if day=="day1":
        #day1
        date = "2025-02-10 00:00:00"
        forecast_end_date = "2025-02-11 00:00:00"
    # day2
    if day=="day2":
        date = "2025-02-20 00:00:00"
        forecast_end_date = "2025-02-21 00:00:00"

    #day 3
    if day=="day3":
        date = "2025-12-21 00:00:00"
        forecast_end_date = "2025-12-22 00:00:00"

    #day4
    if day=="day4":
        date = "2025-07-10 00:00:00"
        forecast_end_date = "2025-07-11 00:00:00"

    #day5
    if day=="day5":
        date = "2025-07-20 00:00:00"
        forecast_end_date = "2025-07-21 00:00:00"
    if day=="day6":
    #day6
        date = "2025-08-10 00:00:00"
        forecast_end_date = "2025-08-11 00:00:00"

    print("now we are doing for day:",day)
    forecast_horizon = 288
    training_size = 288 * 7 * 2
    feature_selection = True
    plot_forecast = True
    hyperparameter_opt = False
    opt_trials = 10

    for number_of_houses in range(first_set_of_house, max_number_of_houses + 1, 10):
        df_original = pd.read_csv(
            path / f"net_load_{number_of_houses}_buildings.csv",
            index_col=0,
            delimiter=",",
            dayfirst=True
        )

        forecasting_households(
            day,
            project_path,
            df_original,
            df_weather,
            date,
            forecast_end_date,
            forecast_horizon,
            training_size,
            feature_selection,
            plot_forecast,
            hyperparameter_opt,
            models,
            number_of_houses,
            opt_trials
        )

        print("the amount of houses that we just did was", number_of_houses)


now we are doing for day: day1

Iteration 1/1
Training XGBoost...
XGBoost training complete.
Training LightGBM...
LightGBM training complete.
Training CatBoost...
CatBoost training complete.
Training RandomForest...
RandomForest training complete.
Training ElasticNet...
ElasticNet training complete.
Training MLP...
MLP training complete.
Training SVR...
SVR training complete.
MSE - XGBoost: 1721063.72237
MSE - LightGBM: 1904623.54467
MSE - CatBoost: 1666406.07749
MSE - RandomForest: 1716640.99979
MSE - ElasticNet: 2241248.24702
MSE - MLP: 1856904.22866
MSE - SVR: 1922836.79218
MSE - Naive Forecast: 3375386.70382
CSV saved at: C:\Users\CR58XM\Desktop\Slovak_case_study\data\outputs\final_predictions_day1_10_houses_and_288_steps.csv
JSON saved at: C:\Users\CR58XM\Desktop\Slovak_case_study\data\outputs\opt_parameters_10_houses_288_steps.json
Plot saved at: C:\Users\CR58XM\Desktop\Slovak_case_study\data\plots\actual_vs_predictions_10_houses_288_steps.png
the amount of houses that we just di

# end

for 1 home is 218 minutes so 3.5 hours

Germany 28 homes *5 
Ireland 20 homes *5
Portugal 22 homes *5 = 350

350*3.5= 1225 hours = 50 days